In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
import xgboost
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'players.csv').exists()), Path.cwd())
DATA_DIR = PROJECT_DIR / 'data'
MODEL_DIR = PROJECT_DIR / 'models'
REPORT_DIR = PROJECT_DIR / 'reports'
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
LEAGUE = "GB1"
START_DATE = pd.Timestamp("2013-01-01")
TEST_CUTOFF = pd.Timestamp("2023-07-01")
print("Data directory:", DATA_DIR)
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__, "xgboost:", xgboost.__version__)


Data directory: c:\Users\Asus\Desktop\PL project\data
pandas: 2.3.3 scikit-learn: 1.9.1 xgboost: 3.4.1


In [2]:
valuation_columns = [
    "player_id", "date", "market_value_in_eur", "current_club_name",
    "current_club_id", "player_club_domestic_competition_id"
]
vals = pd.read_csv(DATA_DIR / "player_valuations.csv", usecols=valuation_columns, parse_dates=["date"])
vals = vals.loc[
    vals["player_club_domestic_competition_id"].eq(LEAGUE)
    & vals["date"].ge(START_DATE)
    & vals["market_value_in_eur"].gt(0)
].sort_values(["date", "player_id"], kind="mergesort").reset_index(drop=True)

player_columns = [
    "player_id", "date_of_birth", "position", "sub_position", "foot",
    "height_in_cm", "country_of_citizenship"
]
player_info = pd.read_csv(DATA_DIR / "players.csv", usecols=player_columns, parse_dates=["date_of_birth"])
appearance_columns = [
    "appearance_id", "player_id", "date", "goals", "assists",
    "minutes_played", "yellow_cards", "red_cards"
]
apps = pd.read_csv(DATA_DIR / "appearances.csv", usecols=appearance_columns, parse_dates=["date"])
print(f"Eligible Premier League valuations: {len(vals):,} ({vals.date.min().date()} to {vals.date.max().date()})")
print(f"Appearance records loaded: {len(apps):,}")


Eligible Premier League valuations: 28,313 (2013-01-04 to 2026-06-03)
Appearance records loaded: 1,894,350


In [3]:
# Build cumulative appearance totals, then query only strictly before each valuation date.
# The lower-bound query is also strict, producing a clean trailing 365-day window.
performance_columns = ["goals", "assists", "minutes_played", "yellow_cards", "red_cards", "games"]
apps = apps.dropna(subset=["player_id", "date"]).copy()
apps["games"] = 1
apps = apps.sort_values(["date", "player_id", "appearance_id"], kind="mergesort").reset_index(drop=True)
cumulative = apps.groupby("player_id", sort=False)[performance_columns].cumsum()
history = pd.concat([apps[["player_id", "date"]], cumulative], axis=1).sort_values("date", kind="mergesort")
queries = vals[["player_id", "date"]].copy()
queries["_row"] = np.arange(len(queries))

def cumulative_asof(query_dates):
    left = queries[["_row", "player_id"]].copy()
    left["query_date"] = query_dates.to_numpy()
    left = left.sort_values("query_date", kind="mergesort")
    matched = pd.merge_asof(
        left, history, left_on="query_date", right_on="date", by="player_id", direction="backward"
    )
    return matched.set_index("_row")[performance_columns].reindex(queries.index).fillna(0)

one_day = pd.Timedelta(nanoseconds=1)
recent = cumulative_asof(vals["date"] - one_day)
prior = cumulative_asof(vals["date"] - pd.Timedelta(days=365) - one_day)
performance = (recent.to_numpy() - prior.to_numpy())
perf = pd.DataFrame(performance, columns=[f"{name}_12m" for name in performance_columns], index=vals.index)
print("Computed strictly prior 365-day performance features.")


Computed strictly prior 365-day performance features.


In [4]:
data = vals.join(perf).merge(player_info, on="player_id", how="left", validate="many_to_one")
data["age"] = (data["date"] - data["date_of_birth"]).dt.days / 365.25
data["minutes_per_game"] = data["minutes_played_12m"] / data["games_12m"].replace(0, np.nan)
data = data.dropna(subset=["age"]).copy()
data["target_log"] = np.log1p(data["market_value_in_eur"])
print(f"Model rows after requiring known age: {len(data):,}")
print(f"Temporal split: train before {TEST_CUTOFF.date()}, test on/after cutoff")


Model rows after requiring known age: 28,304
Temporal split: train before 2023-07-01, test on/after cutoff


In [5]:
numeric_full = ["age", "height_in_cm", "goals_12m", "assists_12m", "minutes_played_12m", "yellow_cards_12m", "red_cards_12m", "games_12m", "minutes_per_game"]
categorical_full = ["position", "sub_position", "foot", "country_of_citizenship", "current_club_name"]
full_features = numeric_full + categorical_full
train = data.loc[data["date"] < TEST_CUTOFF].copy()
test = data.loc[data["date"] >= TEST_CUTOFF].copy()
validation_cutoff = pd.Timestamp("2022-07-01")
selector_train = data.loc[data["date"] < validation_cutoff].copy()
selector_validation = data.loc[(data["date"] >= validation_cutoff) & (data["date"] < TEST_CUTOFF)].copy()
if train.empty or test.empty or selector_train.empty or selector_validation.empty:
    raise ValueError("A chronological train, validation, or test split is empty.")
xgb_parameters = {
    "objective": "reg:squarederror", "tree_method": "hist", "learning_rate": 0.1,
    "max_depth": 4, "min_child_weight": 8, "subsample": 0.85, "colsample_bytree": 0.9,
    "reg_alpha": 0.05, "reg_lambda": 10.0, "n_estimators": 3000,
    "early_stopping_rounds": 100, "eval_metric": "rmse", "n_jobs": 4, "random_state": 42,
}
def make_preprocessor(scale_numeric):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric_full),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32))
        ]), categorical_full)
    ])
print(f"Train: {len(train):,} | early-stop validation: {len(selector_validation):,} | test: {len(test):,}")
print("Test dates:", test["date"].min().date(), "to", test["date"].max().date())


Train: 22,457 | early-stop validation: 3,203 | test: 5,847
Test dates: 2023-07-04 to 2026-06-03


In [6]:
def score_predictions(label, estimator, features):
    predicted_log = estimator.predict(test[features])
    predicted_eur = np.maximum(np.expm1(predicted_log), 0)
    actual_eur = test["market_value_in_eur"].to_numpy()
    absolute_error = np.abs(predicted_eur - actual_eur)
    return {
        "model": label, "rows": int(len(test)),
        "rmse_eur": float(np.sqrt(mean_squared_error(actual_eur, predicted_eur))),
        "mae_eur": float(mean_absolute_error(actual_eur, predicted_eur)),
        "median_absolute_error_eur": float(np.median(absolute_error)),
        "median_absolute_percentage_error": float(np.median(absolute_error / actual_eur) * 100),
        "rmse_log": float(np.sqrt(mean_squared_error(test["target_log"], predicted_log))),
        "r2_log": float(r2_score(test["target_log"], predicted_log)),
    }

# Linear regression is the baseline and uses the same input features as XGBoost.
linear_baseline = Pipeline([
    ("preprocess", make_preprocessor(scale_numeric=True)),
    ("regression", LinearRegression())
])
linear_baseline.fit(train[full_features], train["target_log"])
linear_metrics = score_predictions("Linear regression baseline (same features)", linear_baseline, full_features)

# Select the number of boosting rounds using only a pre-test chronological validation period.
selector_preprocessor = make_preprocessor(scale_numeric=False)
X_selector_train = selector_preprocessor.fit_transform(selector_train[full_features])
X_selector_validation = selector_preprocessor.transform(selector_validation[full_features])
selector_parameters = {key: value for key, value in xgb_parameters.items() if key not in {"n_estimators", "early_stopping_rounds"}}
selector = XGBRegressor(**selector_parameters, n_estimators=xgb_parameters["n_estimators"], early_stopping_rounds=xgb_parameters["early_stopping_rounds"])
selector.fit(X_selector_train, selector_train["target_log"], eval_set=[(X_selector_validation, selector_validation["target_log"])], verbose=False)
best_rounds = int(selector.best_iteration) + 1
print("Early-stopping selected boosting rounds:", best_rounds)

# Refit on all pre-test data; keep the test set untouched until final scoring.
xgb_parameters_final = {key: value for key, value in xgb_parameters.items() if key not in {"early_stopping_rounds", "eval_metric"}}
xgb_test_model = Pipeline([
    ("preprocess", make_preprocessor(scale_numeric=False)),
    ("regression", XGBRegressor(**{**xgb_parameters_final, "n_estimators": best_rounds}))
])
xgb_test_model.fit(train[full_features], train["target_log"])
xgb_metrics = score_predictions("XGBoost primary model", xgb_test_model, full_features)
comparison = [linear_metrics, xgb_metrics]
print(json.dumps(comparison, indent=2))


Early-stopping selected boosting rounds: 1905
[
  {
    "model": "Linear regression baseline (same features)",
    "rows": 5847,
    "rmse_eur": 15655407.721114358,
    "mae_eur": 9723149.946012426,
    "median_absolute_error_eur": 5373983.084157213,
    "median_absolute_percentage_error": 55.63909149225441,
    "rmse_log": 1.032792648392449,
    "r2_log": 0.5299338573383148
  },
  {
    "model": "XGBoost primary model",
    "rows": 5847,
    "rmse_eur": 14423037.346396632,
    "mae_eur": 8590216.0,
    "median_absolute_error_eur": 4315243.0,
    "median_absolute_percentage_error": 47.9313,
    "rmse_log": 0.9315106452705116,
    "r2_log": 0.6176084027320152
  }
]


In [7]:
# Refit the primary XGBoost candidate on all eligible rows after holdout evaluation.
# This is a new artifact; existing notebooks, joblib files, and app models remain untouched.
import joblib
final_xgb = Pipeline([
    ("preprocess", make_preprocessor(scale_numeric=False)),
    ("regression", XGBRegressor(**{**xgb_parameters_final, "n_estimators": best_rounds}))
])
final_xgb.fit(data[full_features], data["target_log"])
artifact = {
    "primary_model": final_xgb, "baseline_model": linear_baseline,
    "numeric_features": numeric_full, "categorical_features": categorical_full,
    "league_competition_id": LEAGUE, "target": "log1p_market_value_in_eur",
    "training_rows": int(len(data)), "training_date_start": str(data["date"].min().date()),
    "training_date_end": str(data["date"].max().date()),
    "performance_window": "strictly prior 365 days",
    "xgboost_parameters": {**xgb_parameters_final, "n_estimators": best_rounds},
    "club_features": "club name from each historical valuation; no current club roster snapshot",
}
joblib.dump(artifact, MODEL_DIR / "pl_value_model_xgboost_retrained.joblib")
report = {
    "primary_model": "xgboost_regressor_log1p",
    "baseline_model": "linear_regression_same_features",
    "league_competition_id": LEAGUE, "test_cutoff": str(TEST_CUTOFF.date()),
    "validation_cutoff": str(validation_cutoff.date()), "train_rows": int(len(train)),
    "validation_rows": int(len(selector_validation)), "test_rows": int(len(test)),
    "test_date_start": str(test["date"].min().date()), "test_date_end": str(test["date"].max().date()),
    "features": full_features, "xgboost_parameters": {**xgb_parameters_final, "n_estimators": best_rounds},
    "metrics": comparison,
    "notes": [
        "XGBoost rounds were selected on a chronological validation window before the test cutoff.",
        "Both algorithms use the same features and chronological test rows.",
        "The report uses the holdout before the primary model is refit on all eligible rows.",
        "RMSE is in euros; median absolute percentage error is in percent.",
        "No current clubs.csv snapshot features are used for historical observations.",
        "Appearance features include only matches strictly before each valuation date.",
    ],
}
(REPORT_DIR / "model_evaluation_xgboost_retrained.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved model: models/pl_value_model_xgboost_retrained.joblib")
print("Saved report: reports/model_evaluation_xgboost_retrained.json")


Saved model: models/pl_value_model_xgboost_retrained.joblib
Saved report: reports/model_evaluation_xgboost_retrained.json
